# House Prices: Advanced Regression Techniques
## Multiple Approaches Comparison

This notebook demonstrates two approaches:
1. **Original Simple Approach**: Linear Regression with 2 features
2. **Advanced Approach**: XGBoost with full feature engineering

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Check for available files
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Load Data

In [ ]:
# Load data
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

## 2. APPROACH 1: Simple Linear Regression (Original Method)

This approach uses only 2 features: MSSubClass and LotFrontage

In [ ]:
# Create subset with only 2 features
train_df_sub = deepcopy(train_df[['MSSubClass', 'LotFrontage', 'SalePrice']])
test_df_sub = deepcopy(test_df[['MSSubClass', 'LotFrontage']])

print("Training subset shape:", train_df_sub.shape)
print("\nMissing values:")
print(train_df_sub.isnull().sum())
train_df_sub.head()

In [ ]:
# Fill missing values with mean
train_df_sub = train_df_sub.fillna(train_df_sub.mean())
test_df_sub = test_df_sub.fillna(train_df_sub[['MSSubClass', 'LotFrontage']].mean())

print("Missing values after imputation:")
print(train_df_sub.isnull().sum())

In [ ]:
# Prepare training data
X_train_simple = train_df_sub.to_numpy()[:, :-1]  # MSSubClass, LotFrontage
y_train_simple = train_df_sub.to_numpy()[:, -1]   # SalePrice
X_test_simple = test_df_sub.to_numpy()

print(f"X_train shape: {X_train_simple.shape}")
print(f"y_train shape: {y_train_simple.shape}")
print(f"X_test shape: {X_test_simple.shape}")

In [ ]:
# Train simple linear regression
lr_simple = LinearRegression()
lr_simple.fit(X_train_simple, y_train_simple)

# Make predictions
train_pred_simple = lr_simple.predict(X_train_simple)
test_pred_simple = lr_simple.predict(X_test_simple)

# Evaluate
rmse_simple = np.sqrt(mean_squared_error(y_train_simple, train_pred_simple))
mae_simple = mean_absolute_error(y_train_simple, train_pred_simple)
r2_simple = r2_score(y_train_simple, train_pred_simple)

print("=" * 60)
print("SIMPLE LINEAR REGRESSION (2 features only)")
print("=" * 60)
print(f"Train RMSE: ${rmse_simple:,.2f}")
print(f"Train MAE: ${mae_simple:,.2f}")
print(f"Train R²: {r2_simple:.4f}")
print("\nCoefficients:")
print(f"  MSSubClass: {lr_simple.coef_[0]:.2f}")
print(f"  LotFrontage: {lr_simple.coef_[1]:.2f}")
print(f"  Intercept: {lr_simple.intercept_:.2f}")

In [ ]:
# Create submission file for simple linear regression
simple_submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': test_pred_simple
})
simple_submission.to_csv('simple_linear_regression.csv', index=False)
print("Simple Linear Regression submission saved!")
simple_submission.head()

## 3. APPROACH 2: Advanced XGBoost with Full Features

Now let's try a more sophisticated approach using all features and gradient boosting

In [ ]:
# Save test IDs and target variable
test_ids = test_df['Id']
train_target = train_df['SalePrice']

# Combine train and test for consistent preprocessing
all_data = pd.concat([train_df.drop('SalePrice', axis=1), test_df], axis=0, ignore_index=True)

print(f"Combined data shape: {all_data.shape}")
print(f"\nMissing values (top 10):")
print(all_data.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Handle missing values
# For numeric features: fill with median
numeric_features = all_data.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_features:
    if all_data[col].isnull().sum() > 0:
        all_data[col].fillna(all_data[col].median(), inplace=True)

# For categorical features: fill with mode or 'None'
categorical_features = all_data.select_dtypes(include=['object']).columns
for col in categorical_features:
    if all_data[col].isnull().sum() > 0:
        all_data[col].fillna('None', inplace=True)

print(f"Missing values after imputation: {all_data.isnull().sum().sum()}")

In [ ]:
# Encode categorical variables using one-hot encoding
all_data_encoded = pd.get_dummies(all_data, drop_first=True)

print(f"Shape after encoding: {all_data_encoded.shape}")
print(f"Number of features: {all_data_encoded.shape[1]}")

In [ ]:
# Split back into train and test
X_train = all_data_encoded.iloc[:len(train_df), :]
X_test = all_data_encoded.iloc[len(train_df):, :]
y_train = train_target

# Drop Id column if it exists
if 'Id' in X_train.columns:
    X_train = X_train.drop('Id', axis=1)
    X_test = X_test.drop('Id', axis=1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

### 3.1 Train XGBoost Model

XGBoost builds trees sequentially where each new tree corrects errors from previous trees. This is typically more powerful than linear models for complex datasets.

In [ ]:
# Install XGBoost if not available
try:
    import xgboost as xgb
    print("XGBoost already installed")
except ImportError:
    print("Installing XGBoost...")
    !pip install xgboost
    import xgboost as xgb
    print("XGBoost installed successfully!")

In [ ]:
# Train XGBoost model
# These parameters are tuned for better performance
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,        # Number of boosting rounds
    learning_rate=0.05,       # Step size shrinkage to prevent overfitting
    max_depth=3,              # Maximum tree depth
    min_child_weight=1,       # Minimum sum of instance weight needed in a child
    subsample=0.8,            # Subsample ratio of training instances
    colsample_bytree=0.8,     # Subsample ratio of columns when constructing each tree
    gamma=0,                  # Minimum loss reduction required to make a split
    reg_alpha=0.1,            # L1 regularization
    reg_lambda=1,             # L2 regularization
    random_state=42
)

print("Training XGBoost model...")
xgb_model.fit(X_train, y_train, 
              eval_set=[(X_train, y_train)],
              verbose=100)

print("\nXGBoost training complete!")

In [ ]:
# Make predictions
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

# Evaluate
xgb_rmse = np.sqrt(mean_squared_error(y_train, xgb_train_pred))
xgb_r2 = r2_score(y_train, xgb_train_pred)
xgb_mae = mean_absolute_error(y_train, xgb_train_pred)

print("XGBoost Performance:")
print(f"Train RMSE: ${xgb_rmse:,.2f}")
print(f"Train R²: {xgb_r2:.4f}")
print(f"Train MAE: ${xgb_mae:,.2f}")

In [ ]:
# Plot top 20 most important features
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
plt.barh(range(len(feature_importance)), feature_importance['importance'])
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Most Important Features (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

In [ ]:
# Create submission file for XGBoost
xgb_submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': xgb_test_pred
})
xgb_submission.to_csv('xgboost_submission.csv', index=False)
print("XGBoost submission saved!")

## 4. Model Comparison: Simple vs Advanced

In [ ]:
# Compare both models
comparison = pd.DataFrame({
    'Model': ['Simple LR (2 features)', 'XGBoost (all features)'],
    'Features Used': [2, X_train.shape[1]],
    'Train RMSE': [rmse_simple, xgb_rmse],
    'Train R²': [r2_simple, xgb_r2],
    'Train MAE': [mae_simple, xgb_mae]
})

comparison = comparison.sort_values('Train RMSE')
print("\n" + "="*90)
print("MODEL COMPARISON (sorted by RMSE - lower is better)")
print("="*90)
print(comparison.to_string(index=False))
print("\n" + "="*90)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['Train RMSE', 'Train R²', 'Train MAE']
colors = ['#1f77b4', '#2ca02c']

for idx, metric in enumerate(metrics):
    axes[idx].bar(comparison['Model'], comparison[metric], color=colors)
    axes[idx].set_title(metric, fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].tick_params(axis='x', rotation=15)
    
plt.tight_layout()
plt.show()

# Show improvement
rmse_improvement = ((rmse_simple - xgb_rmse) / rmse_simple) * 100
print(f"\nRMSE Improvement: {rmse_improvement:.1f}%")
print(f"The advanced XGBoost model reduced error by ${rmse_simple - xgb_rmse:,.2f}")

## 5. Key Insights & Recommendations

**Comparison of Approaches:**

1. **Simple Linear Regression (2 features)**:
   - Uses only MSSubClass and LotFrontage
   - Very simple and interpretable
   - Limited performance due to missing information from other features
   - Good baseline to understand feature impact

2. **XGBoost (all features)**:
   - Uses all available features after preprocessing
   - Captures complex non-linear relationships automatically
   - Handles feature interactions naturally
   - Significantly better performance

**Why XGBoost performs better:**
- Leverages information from ALL features, not just 2
- Captures non-linear patterns (e.g., price doesn't increase linearly with lot size)
- Automatically handles feature interactions (e.g., neighborhood + square footage)
- Built-in regularization prevents overfitting
- Less sensitive to outliers

**Next Steps for Even Better Performance:**
- Feature engineering: create interaction terms, polynomial features
- Hyperparameter tuning using GridSearchCV or Optuna
- Ensemble methods: stack multiple models together
- Cross-validation for more robust evaluation
- Target transformation: log-transform SalePrice to handle skewness

In [ ]:
# Show which submission to use
best_model = comparison.iloc[0]['Model']
best_rmse = comparison.iloc[0]['Train RMSE']

print("\n" + "="*90)
print(f"RECOMMENDED SUBMISSION: {best_model}")
print(f"Best Train RMSE: ${best_rmse:,.2f}")
print("="*90)

if 'XGBoost' in best_model:
    print("\nℹ️  Submit: xgboost_submission.csv")
    print("   This model uses all features and advanced machine learning")
else:
    print("\nℹ️  Submit: simple_linear_regression.csv")
    print("   This model uses only 2 features (MSSubClass, LotFrontage)")
    
print("\n" + "="*90)
print("SUMMARY")
print("="*90)
print(f"Simple Linear Regression RMSE: ${rmse_simple:,.2f}")
print(f"XGBoost RMSE: ${xgb_rmse:,.2f}")
print(f"Improvement: {rmse_improvement:.1f}%")
print("\nConclusion: Using more features and advanced algorithms significantly")
print("improves prediction accuracy!")